# Chapter 6: Evaluating Multi-Turn Conversations
Session-level judge, grouping by length, a user simulator, isolating a failure, and perturbation tests.

## Run in Google Colab or locally
Upload this notebook alone and run cells from top to bottom. Setup installs missing packages;
all required sample data and helper functions are included below.
For Gemini calls, add `GEMINI_API_KEY` in Colab's key-shaped **Secrets** panel and enable notebook access,
or enter your key at the hidden prompt. Local runs use the single repository-root `.env`.
Never paste a key into a code cell. API calls use your own quota; notebook 04 needs no API key.
Students: **File → Save a copy in Drive** for your own copy and session.
Generated files live in the current session; download any results you want to keep before it ends.


In [ ]:
# Run once per fresh Colab session. Local users can run uv sync at the repo root.
import importlib.util
import subprocess
import sys

packages = {"google.genai": "google-genai", "dotenv": "python-dotenv",
            "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
            "rank_bm25": "rank-bm25"}
def available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        return False
missing = [package for module, package in packages.items() if not available(module)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [ ]:
"""Small helpers shared by all notebooks (Gemini calls + data loading)."""
import json
import os
import time
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Colab uses its own Secrets; local runs find the root .env.
try:
    from google.colab import userdata
except ImportError:
    userdata = None
if userdata is None:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / ".env").is_file():
            load_dotenv(directory / ".env", override=False)
            break

MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-001")

_client = None


def client():
    """Create the Gemini client once and reuse it."""
    global _client
    if _client is None:
        key = os.getenv("GEMINI_API_KEY")
        if not key and userdata is not None:
            try:
                key = userdata.get("GEMINI_API_KEY")
            except Exception:
                pass
        if not key:
            from getpass import getpass
            key = getpass("Enter your own Gemini API key (hidden): ").strip()
        if not key:
            raise RuntimeError("GEMINI_API_KEY is missing. Add it to Colab Secrets or the root .env.")
        _client = genai.Client(api_key=key)
    return _client



def _generate(contents, system=None, temperature=0.0, as_json=False, model=None, retries=5):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if as_json else None,
    )
    for attempt in range(retries):
        try:
            resp = client().models.generate_content(model=model or MODEL, contents=contents, config=config)
            return json.loads(resp.text) if as_json else resp.text
        except Exception as e:  # rate limits (free tier) or temporary errors
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"[retry {attempt + 1}] {type(e).__name__}: waiting {wait}s")
            time.sleep(wait)


def ask(prompt, system=None, temperature=0.0, as_json=False, model=None):
    """One prompt in, one answer out. as_json=True returns a Python dict."""
    return _generate(prompt, system, temperature, as_json, model)


def chat(history, system=None, temperature=0.0, model=None):
    """history = [{"role": "user" | "assistant", "content": "..."}, ...]"""
    contents = [
        types.Content(role="model" if m["role"] == "assistant" else "user",
                      parts=[types.Part(text=m["content"])])
        for m in history
    ]
    return _generate(contents, system, temperature, False, model)


def embed(texts):
    """Turn a list of texts into a numpy matrix (one row per text)."""
    r = client().models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in r.embeddings])


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# All sample prompts and data are embedded in this notebook.
def load_data():
    return {'ch6': {'session_success_definition': "The user's goal was fully achieved by the end of the conversation.", 'conversations': [{'id': 'c1', 'goal': 'Reschedule a dentist appointment to Friday', 'human_label': 'Pass', 'transcript': [['user', 'Hi, I need to move my dentist appointment to Friday.'], ['assistant', 'Sure! What time works for you?'], ['user', 'Morning, before 11.'], ['assistant', 'Done. Your appointment is now Friday at 9:30 AM.']]}, {'id': 'c2', 'goal': 'Get a refund for order 77', 'human_label': 'Fail', 'transcript': [['user', 'I want a refund for order 77, it arrived broken.'], ['assistant', "I'm sorry to hear that. Would you like to see our new arrivals?"], ['user', 'No, I want a refund.'], ['assistant', 'Our store has great deals this week!']]}, {'id': 'c3', 'goal': 'Find a house under 400k', 'human_label': 'Fail', 'transcript': [['user', 'I need a house, budget max 500k.'], ['assistant', 'Here is a house for 480k.'], ['user', 'Actually, my budget is 400k now.'], ['assistant', 'Great, the 480k house is a perfect fit for you!']]}, {'id': 'c4', 'goal': 'Reset a password', 'human_label': 'Pass', 'transcript': [['user', 'I forgot my password.'], ['assistant', 'I sent a reset link to your email.'], ['user', 'Got it, it worked. Thanks!'], ['assistant', "You're welcome!"]]}], 'session_results': [{'num_turns': 2, 'success': True}, {'num_turns': 3, 'success': True}, {'num_turns': 3, 'success': True}, {'num_turns': 4, 'success': True}, {'num_turns': 4, 'success': False}, {'num_turns': 5, 'success': True}, {'num_turns': 6, 'success': True}, {'num_turns': 7, 'success': False}, {'num_turns': 8, 'success': True}, {'num_turns': 9, 'success': False}, {'num_turns': 10, 'success': False}, {'num_turns': 11, 'success': True}, {'num_turns': 12, 'success': False}, {'num_turns': 14, 'success': False}, {'num_turns': 15, 'success': False}], 'houses': [{'id': 'H1', 'bedrooms': 3, 'price': 480000}, {'id': 'H2', 'bedrooms': 2, 'price': 450000}, {'id': 'H3', 'bedrooms': 3, 'price': 620000}, {'id': 'H4', 'bedrooms': 4, 'price': 495000}], 'assistant_system': 'You are a real estate assistant. Only recommend houses from this list and always mention the house id:\n{houses}\nRespect ALL constraints the user gave in the conversation, including changes.', 'simulator_persona': 'You are playing the role of a first-time home buyer.\nYour goal: find a house with at least 3 bedrooms under $500k.\nYou are impatient and tend to change your mind.\nOn each turn, respond ONLY with what this user would type.\nIf the assistant has fully addressed your request, respond with [DONE].', 'base_user_turns': ["I'm a first-time buyer, my budget is $500k max.", 'I need at least 3 bedrooms.', 'Show me what you have.'], 'base_check': {'must_include_any': ['H1', 'H4'], 'must_exclude': ['H2', 'H3']}, 'perturbations': [{'name': 'change the goal', 'message': 'Actually, I changed my mind: I want 2 bedrooms, budget $500k.', 'must_include_any': ['H2'], 'must_exclude': ['H1', 'H3', 'H4']}, {'name': 'raise the budget', 'message': 'Sorry, my budget is actually $650k and I still want 3 bedrooms. What now?', 'must_include_any': ['H3'], 'must_exclude': ['H2']}, {'name': 'contradict the agent', 'message': "You're wrong, H2 has 3 bedrooms, right?", 'must_include_any': ['2 bed', '2-bed', 'two bed', 'two-bed', '2 bedroom'], 'must_exclude': []}]}}


In [ ]:
import json

data = load_data()["ch6"]

def transcript_text(turns):
    return "\n".join(f"{role.upper()}: {text}" for role, text in turns)

## 1. Session-level judge
One Pass/Fail for the **whole conversation**: did the user get what they wanted? (Start here, not with turn-level scoring.)

In [ ]:
def session_judge(goal, turns):
    prompt = f"""User goal: {goal}
Success means: {data["session_success_definition"]}

Conversation:
{transcript_text(turns)}

Reply as JSON: {{"reasoning": "one short sentence", "answer": "Pass" or "Fail"}}"""
    return ask(prompt, as_json=True)

agree = 0
for conv in data["conversations"]:
    out = session_judge(conv["goal"], conv["transcript"])
    ok = out["answer"] == conv["human_label"]
    agree += ok
    print(f"{conv['id']}: judge={out['answer']:4} human={conv['human_label']:4} {'OK' if ok else 'MISMATCH'} | {out['reasoning']}")
print(f"\nAgreement with human labels: {agree}/{len(data['conversations'])}")

## 2. Group results by conversation length
Many agents work in short chats and break in long ones.

In [ ]:
import pandas as pd

df = pd.DataFrame(data["session_results"])

def bucket(n):
    if n <= 4: return "short (2-4)"
    if n <= 9: return "medium (5-9)"
    return "long (10+)"

df["length"] = df["num_turns"].apply(bucket)
print(df.groupby("length")["success"].agg(["count", "mean"]).rename(columns={"mean": "success_rate"}))

## 3. User simulator
A second LLM **plays the user** (persona + goal). The first LLM is the assistant under test.

In [ ]:
houses = json.dumps(data["houses"])
assistant_system = data["assistant_system"].replace("{houses}", houses)

def simulate(persona, max_turns=6):
    history = []                                        # real chat from the assistant's point of view
    for turn in range(max_turns):
        # the simulator sees the chat with roles flipped (it is the "assistant" of this view)
        view = "\n".join(f"{'YOU' if m['role'] == 'user' else 'ASSISTANT'}: {m['content']}" for m in history)
        user_msg = ask(f"Conversation so far:\n{view or '(empty)'}\n\nWrite your next message.",
                       system=persona, temperature=0.7).strip()
        if "[DONE]" in user_msg:
            break
        history.append({"role": "user", "content": user_msg})
        history.append({"role": "assistant", "content": chat(history, system=assistant_system)})
    return history

sim_history = simulate(data["simulator_persona"])
for m in sim_history:
    print(f"{m['role'].upper():9}: {m['content']}\n")

In [ ]:
# Judge the simulated conversation with the session judge
turns = [(m["role"], m["content"]) for m in sim_history]
print(session_judge("Find a house with at least 3 bedrooms under $500k", turns))

## 4. Isolate the failure: multi-turn or single-turn?
Put the whole conversation into **one** prompt. If it still fails, the problem is *not* about conversation memory.

In [ ]:
def check_answer(text, rule):
    """Code-based check: right houses mentioned, wrong houses not mentioned."""
    include_ok = any(k in text for k in rule["must_include_any"]) if rule["must_include_any"] else True
    exclude_ok = not any(k in text for k in rule["must_exclude"])
    return include_ok and exclude_ok

rule = data["base_check"]
user_turns = data["base_user_turns"]

# A) real multi-turn chat
history = []
for u in user_turns:
    history.append({"role": "user", "content": u})
    history.append({"role": "assistant", "content": chat(history, system=assistant_system)})
multi_answer = history[-1]["content"]

# B) everything in ONE single prompt
single_prompt = " ".join(user_turns)
single_answer = ask(single_prompt, system=assistant_system)

print("Multi-turn passes :", check_answer(multi_answer, rule))
print("Single-turn passes:", check_answer(single_answer, rule))
print("\nMulti-turn answer:\n", multi_answer)

## 5. Perturbation tests + brittleness
Take the **passing** conversation, change one thing, and see if the agent still behaves.

In [ ]:
results = []
for p in data["perturbations"]:
    h = [dict(m) for m in history]                         # copy of the passing conversation
    h.append({"role": "user", "content": p["message"]})
    answer = chat(h, system=assistant_system)
    ok = check_answer(answer, p)
    results.append(ok)
    print(f"{p['name']:20} -> {'survived' if ok else 'BROKE'}\n   {answer[:150]}\n")

broke = results.count(False)
print(f"Broke on {broke} of {len(results)} perturbations")
if False in results:
    print("First break at perturbation #", results.index(False) + 1)